# Style selection: reconstruction accuracy

Four styles, the complete 4×4 PG/TG matrix, 90 titles (30 per domain),
two prompt seeds and two image seeds. Set the four completed jobs, restart
the kernel and **Run All**.

The figures separate strict and title-aware image verification and show strict
and normalized title matching for each policy. All use the **full planned denominator**;
rejections, failed checks and missing predictions count as zero. Intervals are 10,000
whole-title bootstrap resamples, domain-stratified, with all PG/TG cells kept
together (seed 20260829, pointwise 95%). Overall results give each cell equal weight.

This planned comparison reports every style rather than selecting a winner
post hoc. It uses the same final title and seed grid for all four styles.
Figures appear inline and are saved as PNG/PDF;
CSV diagnostics and provenance are exported without cluttering the notebook.

For compact analysis copies, set `ANALYSIS_METADATA_ONLY=1`. This skips only
the local image-file existence check, not stored verification, errors or planned
observations. No pixels are read by this statistical report. Original images
remain necessary for visual inspection and full archiving. The selected check
policy is recorded in the report manifest.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from semantic_roundtrip.analysis.plotting import (
    heatmap,
    paired_style_analysis,
    save_figure,
    style_accuracy_panel,
)
from semantic_roundtrip.analysis.reporting import (
    DOMAINS,
    QG,
    export_tables,
    load_style_jobs,
    prompt_verifications,
    style_accuracy,
    technical_tables,
    write_manifest,
)

# Silence only the known pandas deprecation; data/errors are not suppressed.
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*",
)

# Run from the repository root or its notebooks/ directory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sns.set_theme(
    style="whitegrid",
    context="notebook",
    rc={"pdf.fonttype": 42, "ps.fonttype": 42, "axes.unicode_minus": False},
)
from semantic_roundtrip.analysis.style_report import (
    STYLE_ANALYSIS_METHODS,
    STYLE_BOOTSTRAP,
    STYLE_PRIMARY_DEFINITION,
    export_style_summary,
    style_centrality,
    validate_style_grid,
)


In [ ]:
# Use a short Windows path for compact copies, e.g. C:/sr/style/<job-directory>.
METADATA_ONLY = os.getenv("ANALYSIS_METADATA_ONLY", "0")
if METADATA_ONLY not in {"0", "1"}:
    raise ValueError("ANALYSIS_METADATA_ONLY must be 0 or 1.")
REQUIRE_IMAGE_FILES = METADATA_ONLY == "0"
print("Local image-file checks:", "required" if REQUIRE_IMAGE_FILES else "omitted (metadata-only copy)")
FREE_JOB = os.getenv("FREE_JOB", "/absolute/path/to/final-direct-core")
SKETCH_JOB = os.getenv("SKETCH_JOB", "/absolute/path/to/final-direct-sketch")
COMIC_JOB = os.getenv("COMIC_JOB", "/absolute/path/to/final-direct-comic")
PHOTOREALISTIC_JOB = os.getenv(
    "PHOTOREALISTIC_JOB", "/absolute/path/to/final-direct-photorealistic"
)
STYLES = {
    "Unrestricted": FREE_JOB,
    "Sketch": SKETCH_JOB,
    "Comic": COMIC_JOB,
    "Photorealistic": PHOTOREALISTIC_JOB,
}
OUTPUT_DIR = (
    Path(os.getenv("OUTPUT_DIR", ROOT / "notebooks/results/style_decision"))
    .expanduser()
    .resolve()
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
jobs, observations, titles = load_style_jobs(STYLES, require_image_files=REQUIRE_IMAGE_FILES)
title_scores = pd.concat(titles.values(), ignore_index=True)
validate_style_grid(title_scores)
accuracy = style_accuracy(title_scores)
centrality = style_centrality(title_scores)

## Overall and by domain

How does reconstruction differ between styles? Points show accuracy; bars show
pointwise 95% intervals. The primary series combines strict image verification
with Strict Exact Match; the other three series are sensitivity analyses.
Unrestricted generation is the least-intervention reference recommendation; the
supervisor confirms the common style after reviewing the complete evidence.


In [ ]:
domains = ["all", *DOMAINS]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True, layout="constrained")
for domain, ax in zip(domains, axes):
    style_accuracy_panel(
        ax, accuracy[accuracy.model_pair.eq("all") & accuracy.domain.eq(domain)], STYLES
    )
    ax.set_title("Overall" if domain == "all" else domain.title())
axes[0].set_ylabel("End-to-end accuracy (%), pointwise 95% CI")
axes[-1].legend(loc="upper right", fontsize=8)
save_figure(
    fig,
    OUTPUT_DIR / "style_accuracy_overall_domains",
    "Style comparison: End-to-end accuracy by domain",
)

## Complete PG/TG matrices

These heatmaps show whether the pooled style result hides PG/TG-specific patterns.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.4), layout="constrained")
for (style, scores), ax in zip(titles.items(), axes):
    image = heatmap(ax, scores, QG, style)
fig.colorbar(image, ax=list(axes), label="End-to-end Strict Exact Match (%)")
save_figure(
    fig,
    OUTPUT_DIR / "style_accuracy_pg_bi_matrices",
    "Style comparison: complete direct PG/TG matrices",
)

## Paired differences from unrestricted generation

Each comparison uses the same title, PG/TG cell and seed combination. The third
heatmap is the style accuracy minus unrestricted accuracy in percentage points.

In [ ]:
style_effects = {}
for style in ["Photorealistic", "Sketch", "Comic"]:
    style_effects[style] = paired_style_analysis(
        titles[style],
        titles["Unrestricted"],
        style,
        "Unrestricted",
        style.lower(),
        OUTPUT_DIR,
    )

## Supporting data

Exports include planned counts, coverage, all three verification decisions, errors
and timings. Prompt decisions are stored once per prompt; image decisions are stored
once per image and policy. `manifest.json` records methods, inputs and code.
Prompt counts deduplicate shared source prompts across inherited TG cells. The centrality
CSV reports mean absolute paired title/model differences from the other styles, with equal
weights. Exact ties are flagged with descriptive ranks and tie counts. It never automatically selects the highest, lowest or central
style. `style_summary.json` freezes methods, source identities and artifact hashes for
the compact import into `final_study.ipynb`.


In [ ]:
technical = technical_tables(observations, titles, list(jobs.items()))
export_tables(
    {
        **technical,
        **{f"style_effects_{name.lower()}": table for name, table in style_effects.items()},
        "style_accuracy": accuracy,
        "style_centrality": centrality,
        "style_title_scores": title_scores,
        "style_prompt_verifications": prompt_verifications(
            pd.concat(observations.values(), ignore_index=True)
        ),
    },
    OUTPUT_DIR,
)
write_manifest(
    ROOT / "notebooks/style_decision.ipynb",
    STYLES,
    OUTPUT_DIR,
    analysis={
        "purpose": "complete_style_comparison",
        "local_image_file_check": "required" if REQUIRE_IMAGE_FILES else "omitted_metadata_only",
        "seed_observations_per_title": 4,
        "model_pairs": [f"{pg}/{bi}" for pg in QG for bi in QG],
        "denominator": "all planned direct observations for every verification and title-match policy",
        "primary_metric": "end_to_end_strict_accuracy",
        "primary_definition": STYLE_PRIMARY_DEFINITION,
        "style_analysis_methods": STYLE_ANALYSIS_METHODS,
        "bootstrap": STYLE_BOOTSTRAP,
    },
)
export_style_summary(OUTPUT_DIR)
